# Libreries

In [1]:
import tensorflow as tf
import pandas as pd
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, Input, BatchNormalization, Flatten
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import regularizers
import numpy as np
from itertools import product

In [2]:
from tensorflow.keras.applications import (
    VGG19,
    ResNet50,
    DenseNet121
)

from tensorflow.keras.applications import (
    vgg19,
    resnet50,
    densenet
)

In [3]:
import google.colab
google.colab.drive.mount('/content/drive')

Mounted at /content/drive


# Definitions


In [4]:
path = "/content/drive/My Drive/Investigacion/UTN/GIBIO/Datasets/Dataset_smpl" #directory of the data
save_path= "/content/drive/My Drive/Investigacion/UTN/GIBIO/Results"
batch_size=350
train_size=0.7
val_size=0.15

## Load dataset

In [5]:
column_names = [
    "chest_circ", "waist_circ", "pelvis_circ", "neck_circ", "bicep_circ", "thigh_circ",
    "knee_circ", "arm_length", "leg_length", "calf_length", "head_circ", "wrist_circ",
    "arm_span", "shoulders_width", "torso_length", "inner_leg"
]

male_data = pd.read_csv(path + "/Annotations/bodymeasurements_m.csv", names=column_names)
female_data = pd.read_csv(path + "/Annotations/bodymeasurements_f.csv", names=column_names)
male_data.insert(0, "ID", [f"men/{i:06d}.png" for i in range(len(male_data))])
female_data.insert(0, "ID", [f"women/{i:06d}.png" for i in range(len(female_data))])

In [6]:
male_data.head(5)

,ID,chest_circ,waist_circ,pelvis_circ,neck_circ,bicep_circ,thigh_circ,knee_circ,arm_length,leg_length,calf_length,head_circ,wrist_circ,arm_span,shoulders_width,torso_length,inner_leg
0,men/000000.png,83.576515,73.213088,92.503097,36.068310,22.853661,46.370356,35.224242,55.186850,84.096402,43.584922,53.987637,15.808483,182.0555,34.255001,49.249297,77.946905
1,men/000001.png,101.702204,88.686590,100.846918,38.591906,28.510385,53.521642,37.731078,49.606380,73.995692,38.563633,57.345473,15.794436,171.1829,36.272103,51.644093,67.475005
2,men/000002.png,104.289594,97.596672,100.898346,43.369495,28.740368,50.741101,37.595967,47.996426,71.371871,36.863151,60.638029,17.591052,170.2976,37.544197,50.525796,67.803296
3,men/000003.png,118.734385,117.314673,118.804251,47.740865,38.573465,58.505852,43.587228,52.009726,77.913660,41.354027,60.986385,20.157620,186.8205,42.696899,57.420403,70.802406
4,men/000004.png,104.736869,93.577234,106.010664,40.026016,33.198243,56.753582,37.691793,49.763331,73.619604,38.421023,58.549413,17.662700,176.7890,40.444499,51.284504,67.094300


In [7]:
preprocess_input = {
    "VGG19": vgg19.preprocess_input,
    "ResNet50": resnet50.preprocess_input,
    "DenseNet121": densenet.preprocess_input
}

In [8]:
def preprocess_image(image_path, measurements, source,model):
    # Read the image
    image = tf.io.read_file(source+image_path)
    image = tf.image.decode_png(image, channels=3)

    # Resize the image to match ResNet50 input
    image = tf.image.resize(image, [224, 224])

    # Normalize the image
    image = preprocess_input[model](image)

    return image, measurements

In [9]:
def create_datasets(male_data,female_data,source, batch_size, train_size, val_size,preprocess_model):
    # Calcular cantidades

    source=source+ "/Images/"
    num_total = len(male_data) # misma cantidad que female 50k
    num_train = int(num_total * train_size)
    num_val = int(num_total * val_size)
    num_test = num_total - num_train - num_val

    male_dataset = tf.data.Dataset.from_tensor_slices((male_data["ID"].values,male_data.drop(columns=["ID"]).values))
    male_dataset = male_dataset.shuffle(buffer_size=len(male_data),seed=34,reshuffle_each_iteration=True)
    test_dataset_male = male_dataset.take(num_test)
    male_dataset =  male_dataset.skip(num_test)

    female_dataset = tf.data.Dataset.from_tensor_slices((female_data["ID"].values,female_data.drop(columns=["ID"]).values))
    female_dataset = female_dataset.shuffle(buffer_size=len(female_data),seed=34,reshuffle_each_iteration=True)
    test_dataset_female = female_dataset.take(num_test)
    female_dataset =  female_dataset.skip(num_test)


    # Unir ambos datasets
    combined_dataset = male_dataset.concatenate(female_dataset)
    combined_dataset = combined_dataset.shuffle(buffer_size=(len(male_data) + len(female_data) - 2*num_test),seed=34,reshuffle_each_iteration=True)

    train_dataset = combined_dataset.take(num_train*2)
    val_dataset = combined_dataset.skip(num_train*2).take(num_val*2)


    # Preprocesamiento y batching
    train_dataset = train_dataset.map(lambda x, y: preprocess_image(x, y, source,preprocess_model),
                                      num_parallel_calls=tf.data.AUTOTUNE).batch(batch_size).prefetch(tf.data.AUTOTUNE)

    val_dataset = val_dataset.map(lambda x, y: preprocess_image(x, y, source,preprocess_model),
                                  num_parallel_calls=tf.data.AUTOTUNE).batch(batch_size).prefetch(tf.data.AUTOTUNE)

    test_dataset_male = test_dataset_male.map(lambda x, y: preprocess_image(x, y, source,preprocess_model),
                                    num_parallel_calls=tf.data.AUTOTUNE).batch(batch_size).prefetch(tf.data.AUTOTUNE)

    test_dataset_female = test_dataset_female.map(lambda x, y: preprocess_image(x, y, source,preprocess_model),
                                    num_parallel_calls=tf.data.AUTOTUNE).batch(batch_size).prefetch(tf.data.AUTOTUNE)

    test_dataset_total= test_dataset_male.concatenate(test_dataset_female)

    return train_dataset, val_dataset, test_dataset_male, test_dataset_female, test_dataset_total

# 4. Model

## Training

In [10]:
encoder_names = [
    #VGG19,
    ResNet50,
    #DenseNet121
]

In [11]:
def get_model(encoder_name):

  encoder=encoder_name(weights="imagenet", include_top=False, input_shape=(224, 224, 3))

  x = encoder.output
  x = Flatten()(x)
  encoder = Model(inputs=encoder.input, outputs=x)
  inp = Input(shape=(224, 224, 3))
  x = inp
  encoder.trainable = False
  x = encoder(x)

  # Capas densas adicionales con BatchNorm y Dropout
  x = Dense(1024, activation='relu')(x)
  x = BatchNormalization()(x)

  x = Dense(512, activation='relu')(x)
  x = BatchNormalization()(x)

  x = Dense(128, activation='relu')(x)
  x = BatchNormalization()(x)

  # Capa de salida para regresión (16 salidas)
  output = Dense(16)(x)

  # Crear el modelo final
  model = Model(inputs=inp, outputs=output)

  return model


In [12]:
def predict_measurements_and_evaluate(dataset, model, target_size=(224, 224)):
    mae = tf.keras.losses.MeanAbsoluteError()
    mse = tf.keras.losses.MeanSquaredError()

    y_trues = []
    y_preds = []

    for batch_x, batch_y in dataset:
        preds = model.predict(batch_x, verbose=0)
        y_preds.append(preds)
        y_trues.append(batch_y.numpy())

    y_preds = np.vstack(y_preds)
    y_trues = np.vstack(y_trues)

    col_names = [
        "chest_circ", "waist_circ", "pelvis_circ", "neck_circ", "bicep_circ", "thigh_circ",
        "knee_circ", "arm_length", "leg_length", "calf_length", "head_circ", "wrist_circ",
        "arm_span", "shoulders_width", "torso_length", "inner_leg"
    ]

    all_mae = []
    all_mse = []
    for i in range(y_preds.shape[1]):
        mae_val = mae(y_trues[:, i], y_preds[:, i]).numpy()
        mse_val = mse(y_trues[:, i], y_preds[:, i]).numpy()
        all_mae.append(mae_val)
        all_mse.append(mse_val)
        print(f"MAE for {col_names[i]}: {mae_val}, MSE for {col_names[i]}: {mse_val}")

    mean_mae = np.mean(all_mae)
    mean_mse = np.mean(all_mse)
    print(f"Mean MAE: {mean_mae}, Mean MSE: {mean_mse}")

    return mean_mae, mean_mse

In [14]:
results_df = pd.DataFrame(columns=["mean_mae", "mean_mse"])
for encoder in encoder_names:
    encoder_name=encoder.__name__
    train_generator,validation_generator,test_generator_male,test_generator_female,test_generator_total=create_datasets(male_data,female_data,path, batch_size, train_size, val_size,encoder_name)
    encoder_name=encoder.__name__
    model = get_model(encoder)
    stopping = EarlyStopping(
        monitor="val_loss",
        min_delta=0,
        patience=10,
        verbose=0,
        mode="auto",
        baseline=None,
        restore_best_weights=True
    )
    model.compile(Adam(learning_rate=1e-4),
        loss='mean_absolute_error', metrics=['mean_absolute_percentage_error']
        )


    model.fit(train_generator,
              epochs=100,
              validation_data = validation_generator,
              callbacks=[ stopping],
              )

    model.save(save_path + f'/model_{encoder_name}.keras')
    print(f"backbone model:{encoder_name}")
    print("-" * 40)
    print("Results for male test dataset")
    predict_measurements_and_evaluate(test_generator_male, model, (224,224))
    print("\n")
    print("Results for female test dataset")
    predict_measurements_and_evaluate(test_generator_female, model, (224,224))
    print("\n")
    print("Results for both male and female test datasets")
    mean_mae, mean_mse = predict_measurements_and_evaluate(test_generator_total, model, (224,224))

    row_name = f"model_{encoder_name}"
    results_df.loc[row_name] = [mean_mae, mean_mse]


    print("-" * 40)
    print("\n")

Epoch 1/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 2758s 13s/step - loss: 64.2481 - mean_absolute_percentage_error: 99.9414 - val_loss: 63.9414 - val_mean_absolute_percentage_error: 99.4549
Epoch 2/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 712s 4s/step - loss: 63.9852 - mean_absolute_percentage_error: 99.4140 - val_loss: 63.0180 - val_mean_absolute_percentage_error: 97.3382
Epoch 3/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 234s 1s/step - loss: 63.4837 - mean_absolute_percentage_error: 98.3376 - val_loss: 61.4691 - val_mean_absolute_percentage_error: 94.0297
Epoch 4/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 133s 661ms/step - loss: 62.7207 - mean_absolute_percentage_error: 96.6681 - val_loss: 60.0511 - val_mean_absolute_percentage_error: 91.1465
Epoch 5/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 121s 603ms/step - loss: 61.7498 - mean_absolute_percentage_error: 94.5646 - val_loss: 53.4174 - val_mean_absolute_percentage_error: 77.0418
Epoch 6/100
200/200 ━━━━━━━━━━━━━━━━━━━━ 119s 590ms/step - loss: 60.5137 - mean_absolute_percentage_er

In [ ]:
results_df.to_excel(save_path+"/resultados.xlsx")

In [15]:
results_df

,mean_mae,mean_mse
model_ResNet50,0.640874,0.824705


# Testing